# 07 — Pandas Fundamentals

Pandas adalah library untuk manipulasi data tabular. Hampir setiap proyek ML butuh Pandas untuk baca, bersihkan, dan siapkan data.

**Yang akan lo pelajari:**
- Series dan DataFrame
- Baca dan tulis data
- Seleksi dan filtering
- Sorting dan groupby
- Handling missing values
- Merge dan join
- Statistik dan agregasi

---

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

## 1. Series — Array 1D dengan Label

In [ ]:
# Buat Series
s = pd.Series([85, 92, 78, 95, 88])
print(s)
print(f"dtype: {s.dtype}")
print(f"shape: {s.shape}")

In [ ]:
# Series dengan custom index
nilai = pd.Series(
    [85, 92, 78, 95, 88],
    index=['Budi', 'Ani', 'Ciko', 'Dewi', 'Eko']
)
print(nilai)
print(nilai['Ani'])       # 92 — akses by label
print(nilai[nilai > 85])  # filter

## 2. DataFrame — Tabel 2D

In [ ]:
# Buat DataFrame dari dict
df = pd.DataFrame({
    'nama': ['Budi', 'Ani', 'Ciko', 'Dewi', 'Eko', 'Fani'],
    'umur': [25, 23, 27, 24, 26, 22],
    'kota': ['Jakarta', 'Bandung', 'Surabaya', 'Medan', 'Yogyakarta', 'Bali'],
    'nilai': [85, 92, 78, 95, 88, np.nan],
    'lulus': [True, True, True, True, True, False]
})

print(df)
print(f"\nShape: {df.shape}")

In [ ]:
# Info dasar DataFrame
print(df.info())          # tipe data, missing values
print()
print(df.describe())      # statistik numerik
print()
print(df.head(3))         # 3 baris pertama
print(df.tail(2))         # 2 baris terakhir

## 3. Seleksi Data

In [ ]:
# Pilih kolom
print(df['nama'])                    # Series
print(df[['nama', 'nilai']])         # DataFrame (list of columns)

In [ ]:
# loc — label-based indexing
print(df.loc[0])                     # baris index 0
print(df.loc[0:2, 'nama':'kota'])    # baris 0-2, kolom nama sampai kota
print(df.loc[df['nilai'] > 85])      # filter kondisi

In [ ]:
# iloc — integer-based indexing
print(df.iloc[0])                    # baris pertama (index 0)
print(df.iloc[0:3, 0:2])            # baris 0-2, kolom 0-1
print(df.iloc[-1])                   # baris terakhir

In [ ]:
# Boolean filtering
print(df[df['umur'] > 24])
print(df[(df['umur'] > 23) & (df['nilai'] > 85)])
print(df[df['kota'].isin(['Jakarta', 'Bali'])])

## 4. Manipulasi Data

In [ ]:
# Tambah kolom baru
df['nilai_huruf'] = df['nilai'].apply(
    lambda x: 'A' if x >= 90 else ('B' if x >= 80 else 'C') if pd.notna(x) else 'N/A'
)

df['kategori_umur'] = pd.cut(
    df['umur'],
    bins=[0, 23, 25, 100],
    labels=['Muda', 'Menengah', 'Senior']
)

print(df[['nama', 'nilai', 'nilai_huruf', 'umur', 'kategori_umur']])

In [ ]:
# Rename kolom
df_renamed = df.rename(columns={'nama': 'name', 'umur': 'age'})
print(df_renamed.columns.tolist())

# Drop kolom
df_dropped = df.drop(columns=['kategori_umur', 'nilai_huruf'])
print(df_dropped.columns.tolist())

# Drop baris
df_no_last = df.drop(index=df.index[-1])
print(df_no_last.shape)

## 5. Handling Missing Values

In [ ]:
# Cek missing values
print(df.isnull().sum())              # jumlah NaN per kolom
print(df.isnull().sum().sum())        # total NaN
print(df[df['nilai'].isnull()])       # baris yang ada NaN di kolom nilai

In [ ]:
# Isi NaN
df_filled = df.copy()

# Isi dengan nilai tertentu
df_filled['nilai'] = df_filled['nilai'].fillna(0)
print(df_filled[['nama', 'nilai']])

# Isi dengan mean/median
df2 = df.copy()
df2['nilai'] = df2['nilai'].fillna(df2['nilai'].mean())
print(f"\nDiisi dengan mean ({df['nilai'].mean():.1f}):")
print(df2[['nama', 'nilai']])

In [ ]:
# Drop baris yang ada NaN
df_clean = df.dropna()               # hapus baris dengan NaN manapun
df_clean2 = df.dropna(subset=['nilai'])  # hanya cek kolom nilai

print(f"Original: {len(df)} baris")
print(f"Setelah dropna(): {len(df_clean)} baris")

## 6. Sorting dan Groupby

In [ ]:
# Sorting
df_sorted = df.sort_values('nilai', ascending=False)
print(df_sorted[['nama', 'nilai']])

# Sort multiple kolom
df_multi_sort = df.sort_values(['kota', 'nilai'], ascending=[True, False])
print(df_multi_sort[['nama', 'kota', 'nilai']])

In [ ]:
# GroupBy — agregasi per grup
df_clean3 = df.dropna(subset=['nilai'])

# Rata-rata nilai per status lulus
print(df_clean3.groupby('lulus')['nilai'].mean())

# Multiple agregasi
print(df_clean3.groupby('lulus')['nilai'].agg(['mean', 'min', 'max', 'count']))

# GroupBy multiple kolom
print(df_clean3.groupby(['lulus'])['umur'].mean())

## 7. Merge dan Concat

In [ ]:
# Buat dua DataFrame
df_a = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'nama': ['Budi', 'Ani', 'Ciko', 'Dewi']
})

df_b = pd.DataFrame({
    'id': [1, 2, 3, 5],
    'nilai': [85, 92, 78, 88]
})

# Inner join — hanya yang ada di kedua DF
print("INNER JOIN:")
print(pd.merge(df_a, df_b, on='id', how='inner'))

# Left join — semua dari kiri, match dari kanan
print("\nLEFT JOIN:")
print(pd.merge(df_a, df_b, on='id', how='left'))

In [ ]:
# Concat — gabung DataFrame secara vertikal atau horizontal
df_1 = pd.DataFrame({'nama': ['Budi', 'Ani'], 'nilai': [85, 92]})
df_2 = pd.DataFrame({'nama': ['Ciko', 'Dewi'], 'nilai': [78, 95]})

# Vertikal (tumpuk)
df_combined = pd.concat([df_1, df_2], ignore_index=True)
print(df_combined)

# Horizontal
df_h = pd.concat([df_1, df_2], axis=1)
print(df_h)

## 8. Pivot dan Melt

In [ ]:
# Data penjualan
penjualan = pd.DataFrame({
    'bulan': ['Jan', 'Jan', 'Feb', 'Feb', 'Mar', 'Mar'],
    'produk': ['A', 'B', 'A', 'B', 'A', 'B'],
    'total': [100, 150, 120, 180, 110, 160]
})

# Pivot — jadikan kolom menjadi baris
pivot = penjualan.pivot(index='bulan', columns='produk', values='total')
print("PIVOT TABLE:")
print(pivot)

# Melt — kebalikannya (wide → long format)
melted = pivot.reset_index().melt(id_vars='bulan', var_name='produk', value_name='total')
print("\nMELTED:")
print(melted)

In [ ]:
# LATIHAN AKHIR — Analisis data sederhana
# Buat DataFrame simulasi nilai mahasiswa

np.random.seed(42)
n = 50

mahasiswa = pd.DataFrame({
    'nama': [f'Mahasiswa_{i}' for i in range(n)],
    'jurusan': np.random.choice(['Informatika', 'Manajemen', 'Akuntansi'], n),
    'semester': np.random.randint(1, 9, n),
    'ipk': np.round(np.random.uniform(2.0, 4.0, n), 2)
})

# Pertanyaan:
# 1. Rata-rata IPK per jurusan
print("1. Rata-rata IPK per jurusan:")
print(mahasiswa.groupby('jurusan')['ipk'].mean().round(2))

# 2. Mahasiswa dengan IPK tertinggi per jurusan
print("\n2. IPK tertinggi per jurusan:")
print(mahasiswa.loc[mahasiswa.groupby('jurusan')['ipk'].idxmax()][['jurusan', 'nama', 'ipk']])

# 3. Distribusi jurusan
print("\n3. Distribusi jurusan:")
print(mahasiswa['jurusan'].value_counts())

---
## ➡️ Next
Lanjut ke **[08_data_visualization.ipynb](08_data_visualization.ipynb)** — plot data dengan Matplotlib.